In [2]:
import pandas as pd
import duckdb
data = [
    # User A
    ["A", "2026-07-01 09:00:00", "O001", 100],
    ["A", "2026-07-01 10:30:00", "O002", 80],
    ["A", "2026-07-01 14:00:00", "O003", 120],
    ["A", "2026-07-02 09:20:00", "O004", 60],

    # User B
    ["B", "2026-07-01 09:20:00", "O005", 60],
    ["B", "2026-07-01 11:00:00", "O006", 90],
    ["B", "2026-07-02 16:00:00", "O007", 150],

    # User C
    ["C", "2026-07-01 10:00:00", "O008", 200],
    ["C", "2026-07-01 15:30:00", "O009", 50],
    ["C", "2026-07-02 12:00:00", "O010", 75],
]

df = pd.DataFrame(
    data,
    columns=["user_id", "order_time", "order_id", "amount"]
)

df["order_time"] = pd.to_datetime(df["order_time"])

print(df)



  user_id          order_time order_id  amount
0       A 2026-07-01 09:00:00     O001     100
1       A 2026-07-01 10:30:00     O002      80
2       A 2026-07-01 14:00:00     O003     120
3       A 2026-07-02 09:20:00     O004      60
4       B 2026-07-01 09:20:00     O005      60
5       B 2026-07-01 11:00:00     O006      90
6       B 2026-07-02 16:00:00     O007     150
7       C 2026-07-01 10:00:00     O008     200
8       C 2026-07-01 15:30:00     O009      50
9       C 2026-07-02 12:00:00     O010      75


## 题目要求

### 分别使用 SQL 和 Pandas 完成：

- 计算每个用户每次下单后，当前是该用户的第几次下单。

- 最终输出字段：

    - `user_id`
    - `order_time`
    - `order_id`
    - `amount`
    - `running_order_count`

In [12]:
# SQL轨道

query = """

SELECT
    user_id,
    order_time,
    order_id,
    amount,
    COUNT(*)
        OVER(
            PARTITION BY user_id 
                ORDER BY order_time,order_id
                    ROWS BETWEEN UNBOUNDED PRECEDING
                        AND CURRENT ROW
        ) AS running_order_count
FROM df
ORDER BY user_id,order_time,order_id
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,user_id,order_time,order_id,amount,running_order_count
0,A,2026-07-01 09:00:00,O001,100,1
1,A,2026-07-01 10:30:00,O002,80,2
2,A,2026-07-01 14:00:00,O003,120,3
3,A,2026-07-02 09:20:00,O004,60,4
4,B,2026-07-01 09:20:00,O005,60,1
5,B,2026-07-01 11:00:00,O006,90,2
6,B,2026-07-02 16:00:00,O007,150,3
7,C,2026-07-01 10:00:00,O008,200,1
8,C,2026-07-01 15:30:00,O009,50,2
9,C,2026-07-02 12:00:00,O010,75,3


In [10]:
# PANDAS轨道

df_pd = (
    df
    .sort_values(by=['user_id','order_time','order_id'],ascending=[True,True,True])
    .assign(
        running_order_count =  lambda x:(
            x.groupby('user_id')
            .cumcount() + 1
        )
    )
    .reset_index(drop=True)
)
df_pd

,user_id,order_time,order_id,amount,running_order_count
0,A,2026-07-01 09:00:00,O001,100,1
1,A,2026-07-01 10:30:00,O002,80,2
2,A,2026-07-01 14:00:00,O003,120,3
3,A,2026-07-02 09:20:00,O004,60,4
4,B,2026-07-01 09:20:00,O005,60,1
5,B,2026-07-01 11:00:00,O006,90,2
6,B,2026-07-02 16:00:00,O007,150,3
7,C,2026-07-01 10:00:00,O008,200,1
8,C,2026-07-01 15:30:00,O009,50,2
9,C,2026-07-02 12:00:00,O010,75,3
